In [3]:
# 8 PUZZLE USING A* SEARCH ALGORITHM

import heapq

class PuzzleNode:
    """Represents a state in the 8-puzzle search tree."""
    def __init__(self, board, parent=None, move="", g=0, h=0):
        self.board = board
        self.parent = parent
        self.move = move
        self.g = g
        self.h = h
        self.f = g + h

    def __lt__(self, other):
        return self.f < other.f

def calculate_manhattan_distance(board, goal):
    """Calculates the sum of Manhattan distances of all tiles from the goal."""
    distance = 0
    for i in range(9):
        tile = board[i]
        if tile != 0:  # Skip the empty space
            # Current coordinates
            curr_row, curr_col = divmod(i, 3)
            # Goal coordinates
            goal_idx = goal.index(tile)
            goal_row, goal_col = divmod(goal_idx, 3)

            distance += abs(curr_row - goal_row) + abs(curr_col - goal_col)
    return distance

def get_neighbors(node, goal):
    """Generates valid neighbor nodes by moving the empty tile (0)."""
    neighbors = []
    board = node.board
    empty_idx = board.index(0)
    row, col = divmod(empty_idx, 3)

    # Map possible directional moves: (row_change, col_change, move_name)
    moves = {
        (-1, 0): "Up",
        (1, 0): "Down",
        (0, -1): "Left",
        (0, 1): "Right"
    }

    for (dr, dc), move_name in moves.items():
        new_row, new_col = row + dr, col + dc

        if 0 <= new_row < 3 and 0 <= new_col < 3:
            new_idx = new_row * 3 + new_col
            # Convert tuple to list to swap tiles, then convert back
            new_board = list(board)
            new_board[empty_idx], new_board[new_idx] = new_board[new_idx], new_board[empty_idx]
            new_board_tuple = tuple(new_board)

            g_cost = node.g + 1
            h_cost = calculate_manhattan_distance(new_board_tuple, goal)

            neighbors.append(PuzzleNode(new_board_tuple, node, move_name, g_cost, h_cost))

    return neighbors

def solve_8_puzzle(start_state, goal_state):
    """Solves the 8-puzzle problem using the A* Search algorithm."""
    start_tuple = tuple(start_state)
    goal_tuple = tuple(goal_state)

    # Initialize open list (priority queue) and closed set (visited configurations)
    open_list = []
    start_h = calculate_manhattan_distance(start_tuple, goal_tuple)
    start_node = PuzzleNode(start_tuple, parent=None, move="Start", g=0, h=start_h)
    heapq.heappush(open_list, start_node)

    # Dictionary to keep track of the best g-score for visited states
    visited = {start_tuple: 0}
    nodes_expanded = 0

    while open_list:
        current_node = heapq.heappop(open_list)
        nodes_expanded += 1

        # Check if the goal state is reached
        if current_node.board == goal_tuple:
            return current_node, nodes_expanded

        for neighbor in get_neighbors(current_node, goal_tuple):
            # If the neighbor hasn't been visited or we found a shorter path to it
            if neighbor.board not in visited or neighbor.g < visited[neighbor.board]:
                visited[neighbor.board] = neighbor.g
                heapq.heappush(open_list, neighbor)

    return None, nodes_expanded  # Return None if puzzle is unsolvable

def print_board(board):
    """Utility to print a 1D tuple formatted cleanly as a 3x3 matrix."""
    for i in range(0, 9, 3):
        print(f" {board[i]} {board[i+1]} {board[i+2]} ")
    print("-" * 11)

def print_solution(solution_node, nodes_expanded):
    """Traces back from the goal node to print the entire path sequence."""
    path = []
    current = solution_node
    while current:
        path.append(current)
        current = current.parent
    path.reverse()

    print(f"🎉 Goal reached in {solution_node.g} steps!")
    print(f"🔍 Total nodes expanded: {nodes_expanded}\n")

    for step, node in enumerate(path):
        if step == 0:
            print("Initial State:")
        else:
            print(f"Step {step}: Move {node.move}")
        print_board(node.board)

# --- Execution ---
if __name__ == "__main__":
    # 0 represents the blank space
    # (Note: Standard configurations must be solvable)
    initial_puzzle = [2,8,3,
                      1,6,4,
                      7,0,5]

    target_puzzle  = [1, 2, 3,
                      8,0,4,
                      7,6,5]

    solution, expanded = solve_8_puzzle(initial_puzzle, target_puzzle)

    if solution:
        print_solution(solution, expanded)
    else:
        print("❌ No solution exists for this initial state.")


🎉 Goal reached in 5 steps!
🔍 Total nodes expanded: 6

Initial State:
 2 8 3 
 1 6 4 
 7 0 5 
-----------
Step 1: Move Up
 2 8 3 
 1 0 4 
 7 6 5 
-----------
Step 2: Move Up
 2 0 3 
 1 8 4 
 7 6 5 
-----------
Step 3: Move Left
 0 2 3 
 1 8 4 
 7 6 5 
-----------
Step 4: Move Down
 1 2 3 
 0 8 4 
 7 6 5 
-----------
Step 5: Move Right
 1 2 3 
 8 0 4 
 7 6 5 
-----------


In [10]:
# 4 QUEENS USING HILL CLIMB ALGORITHM

import random

def calculate_cost(state):
    """Calculates the number of attacking queen pairs."""
    attacks = 0
    n = len(state)
    for i in range(n):
        for j in range(i + 1, n):
            if state[i] == state[j] or abs(state[i] - state[j]) == (j - i):
                attacks += 1
    return attacks

def draw_board(state):
    """Draws the chessboard representing the queen positions."""
    n = len(state)
    for row in range(n):
        row_str = ""
        for col in range(n):
            if state[col] == row:
                row_str += " Q "  # Queen
            else:
                row_str += " . "  # Empty slot
        print(row_str)
    print()

def get_neighbors(state):
    """Generates all possible neighboring states by moving one queen in its column."""
    neighbors = []
    n = len(state)
    for col in range(n):
        for row in range(n):
            if state[col] != row:
                neighbor = list(state)
                neighbor[col] = row
                neighbors.append(neighbor)
    return neighbors

def hill_climbing_4_queens(initial_state):
    """
    Runs the Hill Climbing algorithm with sideways moves allowed
    to escape basic local plateaus. Displays state diagrams at each step.
    """
    current_state = list(initial_state)
    current_cost = calculate_cost(current_state)
    step = 0

    print(f"=== Starting Combination: {current_state} (Cost: {current_cost}) ===")
    draw_board(current_state)

    sideways_moves = 0
    max_sideways = 20

    while current_cost > 0:
        neighbors = get_neighbors(current_state)
        neighbor_costs = [(nb, calculate_cost(nb)) for nb in neighbors]

        # Sort neighbors to find the minimum cost
        neighbor_costs.sort(key=lambda x: x[1])
        best_cost = neighbor_costs[0][1]
        best_neighbors = [nb for nb, cost in neighbor_costs if cost == best_cost]

        if best_cost < current_cost:
            current_state = random.choice(best_neighbors)
            current_cost = best_cost
            sideways_moves = 0
            step += 1
            print(f"Step {step}: Moved to {current_state} (Cost: {current_cost})")
            draw_board(current_state)
        elif best_cost == current_cost and sideways_moves < max_sideways:
            current_state = random.choice(best_neighbors)
            current_cost = best_cost
            sideways_moves += 1
            step += 1
            print(f"Step {step} (Sideways): Moved to {current_state} (Cost: {current_cost})")
            draw_board(current_state)
        else:
            print(f"❌ Stuck at local minimum at {current_state} (Cost: {current_cost}).")
            return False

    print(f"✨ Success! Found optimal solution at {current_state} (Cost: 0)")
    return True

def run_solver():
    print("--- 4-Queens Hill Climbing Solver with Visualizations ---")
    print("Enter 4 numbers (0 to 3) representing the row index for each column separated by spaces.")
    print("Example: '1 3 0 2' or '3 1 2 0'")

    # For automation / prompt safety, providing fallback defaults if input is bypassed
    try:
        user_input = input("Enter combination: ").strip().split()
        if len(user_input) != 4:
            raise ValueError("Must provide exactly 4 values.")
        initial_state = [int(x) for x in user_input]
        if any(x < 0 or x > 3 for x in initial_state):
            raise ValueError("Values must be between 0 and 3.")
    except Exception as e:
        initial_state = [3, 1, 2, 0]
        print(f"Using default initial combination: {initial_state}")

    # Attempt solving starting from user input
    success = hill_climbing_4_queens(initial_state)

    # Random restart loop if stuck
    restart_count = 1
    while not success:
        print(f"\n↔ Restarting with a random combination (Attempt #{restart_count})...")
        random_initial = [random.randint(0, 3) for _ in range(4)]
        success = hill_climbing_4_queens(random_initial)
        restart_count += 1

run_solver()

--- 4-Queens Hill Climbing Solver with Visualizations ---
Enter 4 numbers (0 to 3) representing the row index for each column separated by spaces.
Example: '1 3 0 2' or '3 1 2 0'
Enter combination: 3 1 2 0
=== Starting Combination: [3, 1, 2, 0] (Cost: 2) ===
 .  .  .  Q 
 .  Q  .  . 
 .  .  Q  . 
 Q  .  .  . 

Step 1 (Sideways): Moved to [3, 1, 3, 0] (Cost: 2)
 .  .  .  Q 
 .  Q  .  . 
 .  .  .  . 
 Q  .  Q  . 

Step 2: Moved to [2, 1, 3, 0] (Cost: 1)
 .  .  .  Q 
 .  Q  .  . 
 Q  .  .  . 
 .  .  Q  . 

Step 3 (Sideways): Moved to [2, 0, 3, 0] (Cost: 1)
 .  Q  .  Q 
 .  .  .  . 
 Q  .  .  . 
 .  .  Q  . 

Step 4: Moved to [2, 0, 3, 1] (Cost: 0)
 .  Q  .  . 
 .  .  .  Q 
 Q  .  .  . 
 .  .  Q  . 

✨ Success! Found optimal solution at [2, 0, 3, 1] (Cost: 0)
